In [11]:
import os
import subprocess
import pathlib
import re
import fnmatch
from subprocess import Popen
import asyncio
from concurrent.futures import ThreadPoolExecutor
import time

# List the current directory
#print(os.listdir(.))


# creating a 'preemption' scenario

In [12]:
# Ask the user to select a digit in the range 0 to 8
k = int(input("Please select a digit in the range 0 to 8: "))

# Check if the digit is within the valid range
if 0 <= k <= 8:
    print(f"will be running in k{k}")
else:
    print("Invalid selection. Please select a digit between 0 and 8.")
    exit(1)

from pathlib import Path

wd = Path(f"/home/rfriedma/src/k{k}/ceph/build")
os.chdir(wd)
%env CEPH_JTEST_ROOT=/home/rfriedma/src/k{k}/ceph/build
!echo $CEPH_JTEST_ROOT > /tmp/jpath


will be running in k3
env: CEPH_JTEST_ROOT=/home/rfriedma/src/k3/ceph/build


In [13]:
!ls -l
!bash -c MGR=0 ls -l


total 11888
drwxr-xr-x.  2 rfriedma rfriedma   20480 Oct  9 02:34 bin
drwxr-xr-x.  6 rfriedma rfriedma      54 Apr 12 03:23 boost
-rw-r--r--.  1 rfriedma rfriedma 7844725 Oct  9 02:32 build.ninja
-rw-r--r--.  1 rfriedma rfriedma    4969 Oct  9 05:48 ceph.conf
-rw-r--r--.  1 rfriedma rfriedma  100338 Oct  4 11:25 CMakeCache.txt
drwxr-xr-x.  9 rfriedma rfriedma    4096 Oct  9 02:32 CMakeFiles
-rw-r--r--.  1 rfriedma rfriedma    2972 Apr 12 03:19 cmake_install.cmake
-rw-r--r--.  1 rfriedma rfriedma 4124472 Oct  9 02:32 compile_commands.json
-rw-r--r--.  1 rfriedma rfriedma      71 Oct  9 02:32 cpm-package-lock.cmake
-rw-r--r--.  1 rfriedma rfriedma     419 Oct  4 11:25 CTestTestfile.cmake
drwxr-xr-x.  5 rfriedma rfriedma      67 May 21 07:12 _deps
drwxr-xr-x.  8 rfriedma rfriedma      80 Oct  9 05:48 dev
drwxr-xr-x.  3 rfriedma rfriedma      78 Oct  9 02:32 doc
drwxr-xr-x.  3 rfriedma rfriedma      20 Apr 12 03:19 etc
drwxr-xr-x.  2 rfriedma rfriedma    4096 Oct  9 02:32 include
-rw------

In [14]:
%%bash
scrtch=to_"`date +%d_%H%M`"
echo $scrtch

MDS=0 MGR=1 OSD=3 MON=1 ../src/vstart.sh -n  --without-dashboard --msgr2 -X --memstore -o "memstore_device_bytes=68435456" -o "osd_op_queue=wpq"
sleep 2
bin/ceph -s

bin/ceph tell osd.* config set debug_osd 10/10

bin/ceph config set global osd_pool_default_pg_autoscale_mode off
sleep 2

# disable rescheduling of the queue due to 'no-scrub' flags
#bin/ceph tell osd.* config set osd_scrub_backoff_ratio 0.9999

# initial set of global scrub scheduling parameters
bin/ceph tell osd.* config set osd_scrub_interval_randomize_ratio 0.0
bin/ceph tell osd.* config set osd_deep_scrub_randomize_ratio 0
# bin/ceph tell osd.* config set osd_scrub_min_interval 10
# bin/ceph tell osd.* config set osd_scrub_max_interval 2000
# bin/ceph tell osd.* config set osd_deep_scrub_interval 600


#PL1 is of size 3

bin/ceph osd pool create pl1 1 1
#bin/ceph osd pool autoscale-status
sleep 1
pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo $pl1_num
bin/ceph osd pool set pl1 size 3
bin/ceph osd pool set pl1 min_size 3
bin/ceph osd pool set pl1 pg_autoscale_mode off
bin/ceph osd pool stats
#bin/ceph osd pool set pl1 noscrub 0
#bin/ceph osd pool set pl1 nodeep-scrub 0
sleep 2

bin/rados bench -p pl1 -t 1 1 write -b 4096 --max-objects 4  --no-cleanup; 
#bin/rados bench -p pl1 1 write -b 4096 --max-objects 128 --show-time --no-cleanup --run-name eeeee

sleep 1
bin/ceph tell osd.* config set debug_osd 10/10

# the set of PGs
bin/ceph pg dump
bin/ceph pg dump pgs_brief
bin/ceph pg dump pgs_brief -f=json-pretty

to_09_0550


rm -f core* 


hostname o10
ip 172.21.64.10
port 40597


/mnt/nvme0/src/k3/ceph/build/bin/ceph-authtool --create-keyring --gen-key --name=mon. /mnt/nvme0/src/k3/ceph/build/keyring --cap mon 'allow *' 


creating /mnt/nvme0/src/k3/ceph/build/keyring


...

dumped all
*** DEVELOPER MODE: setting PATH, PYTHONPATH and LD_LIBRARY_PATH ***
2025-10-09T05:51:07.645-0500 7f8474f6b6c0 -1 WARNING: all dangerous and experimental features are enabled.
2025-10-09T05:51:07.660-0500 7f8474f6b6c0 -1 WARNING: all dangerous and experimental features are enabled.


PG_STAT  STATE         UP       UP_PRIMARY  ACTING   ACTING_PRIMARY
2.0      active+clean  [2,1,0]           2  [2,1,0]               2
1.0      active+clean  [1,0,2]           1  [1,0,2]               1


dumped pgs_brief
*** DEVELOPER MODE: setting PATH, PYTHONPATH and LD_LIBRARY_PATH ***
2025-10-09T05:51:08.004-0500 7f6b23f166c0 -1 WARNING: all dangerous and experimental features are enabled.
2025-10-09T05:51:08.017-0500 7f6b22c8d6c0 -1 WARNING: all dangerous and experimental features are enabled.



{
    "pg_ready": true,
    "pg_stats": [
        {
            "pgid": "2.0",
            "state": "active+clean",
            "up": [
                2,
                1,
                0
            ],
            "acting": [
                2,
                1,
                0
            ],
            "up_primary": 2,
            "acting_primary": 2
        },
        {
            "pgid": "1.0",
            "state": "active+clean",
            "up": [
                1,
                0,
                2
            ],
            "acting": [
                1,
                0,
                2
            ],
            "up_primary": 1,
            "acting_primary": 1
        }
    ]
}


dumped pgs_brief


In [15]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
jp=$CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    %env CEPH_JTEST_ROOT=$jp
fi
cd $jp


--------> /home/rfriedma/src/k3/ceph/build


In [16]:
%%bash

file_path="/tmp/jpath"
if [ -f "$file_path" ]; then
        file_contents=$(cat "$file_path")
        echo "File contents read into variable."
else
        echo "File does not exist."
fi

File contents read into variable.


In [ ]:
%%bash
nosd=5
pl1_num=1 # RRR
for ((i=0;i<=$nosd;i++)); do
  echo "Query map for OSD $i"
  bin/ceph pg $pl1_num.$i query | jq '.scrubber' 
done


Query map for OSD 0


*** DEVELOPER MODE: setting PATH, PYTHONPATH and LD_LIBRARY_PATH ***
2025-10-09T05:51:08.512-0500 7f6cf8b546c0 -1 WARNING: all dangerous and experimental features are enabled.
2025-10-09T05:51:08.526-0500 7f6cf8b546c0 -1 WARNING: all dangerous and experimental features are enabled.
no valid command found; 10 closest matches:
pg stat
pg getmap
pg dump [<dumpcontents:all|summary|sum|delta|pools|osds|pgs|pgs_brief>...]
pg dump_json [<dumpcontents:all|summary|sum|pools|osds|pgs>...]
pg dump_pools_json
pg ls-by-pool <poolstr> [<states>...]
pg ls-by-primary <id|osd.id> [<pool:int>] [<states>...]
pg ls-by-osd <id|osd.id> [<pool:int>] [<states>...]
pg ls [<pool:int>] [<states>...]
pg dump_stuck [<stuckops:inactive|unclean|stale|undersized|degraded>...] [<threshold:int>]
Error EINVAL: invalid command


Query map for OSD 1


*** DEVELOPER MODE: setting PATH, PYTHONPATH and LD_LIBRARY_PATH ***
2025-10-09T05:51:08.868-0500 7fcd085ea6c0 -1 WARNING: all dangerous and experimental features are enabled.
2025-10-09T05:51:08.882-0500 7fcd085ea6c0 -1 WARNING: all dangerous and experimental features are enabled.
no valid command found; 10 closest matches:
pg stat
pg getmap
pg dump [<dumpcontents:all|summary|sum|delta|pools|osds|pgs|pgs_brief>...]
pg dump_json [<dumpcontents:all|summary|sum|pools|osds|pgs>...]
pg dump_pools_json
pg ls-by-pool <poolstr> [<states>...]
pg ls-by-primary <id|osd.id> [<pool:int>] [<states>...]
pg ls-by-osd <id|osd.id> [<pool:int>] [<states>...]
pg ls [<pool:int>] [<states>...]
pg dump_stuck [<stuckops:inactive|unclean|stale|undersized|degraded>...] [<threshold:int>]
Error EINVAL: invalid command


Query map for OSD 2


...

*** DEVELOPER MODE: setting PATH, PYTHONPATH and LD_LIBRARY_PATH ***
2025-10-09T05:51:09.570-0500 7f2a5696b6c0 -1 WARNING: all dangerous and experimental features are enabled.
2025-10-09T05:51:09.585-0500 7f2a5696b6c0 -1 WARNING: all dangerous and experimental features are enabled.
no valid command found; 10 closest matches:
pg stat
pg getmap
pg dump [<dumpcontents:all|summary|sum|delta|pools|osds|pgs|pgs_brief>...]
pg dump_json [<dumpcontents:all|summary|sum|pools|osds|pgs>...]
pg dump_pools_json
pg ls-by-pool <poolstr> [<states>...]
pg ls-by-primary <id|osd.id> [<pool:int>] [<states>...]
pg ls-by-osd <id|osd.id> [<pool:int>] [<states>...]
pg ls [<pool:int>] [<states>...]
pg dump_stuck [<stuckops:inactive|unclean|stale|undersized|degraded>...] [<threshold:int>]
Error EINVAL: invalid command


Query map for OSD 4


*** DEVELOPER MODE: setting PATH, PYTHONPATH and LD_LIBRARY_PATH ***
2025-10-09T05:51:09.917-0500 7f7a42b166c0 -1 WARNING: all dangerous and experimental features are enabled.
2025-10-09T05:51:09.931-0500 7f7a42b166c0 -1 WARNING: all dangerous and experimental features are enabled.
no valid command found; 10 closest matches:
pg stat
pg getmap
pg dump [<dumpcontents:all|summary|sum|delta|pools|osds|pgs|pgs_brief>...]
pg dump_json [<dumpcontents:all|summary|sum|pools|osds|pgs>...]
pg dump_pools_json
pg ls-by-pool <poolstr> [<states>...]
pg ls-by-primary <id|osd.id> [<pool:int>] [<states>...]
pg ls-by-osd <id|osd.id> [<pool:int>] [<states>...]
pg ls [<pool:int>] [<states>...]
pg dump_stuck [<stuckops:inactive|unclean|stale|undersized|degraded>...] [<threshold:int>]
Error EINVAL: invalid command


Query map for OSD 5


*** DEVELOPER MODE: setting PATH, PYTHONPATH and LD_LIBRARY_PATH ***
2025-10-09T05:51:10.263-0500 7f460efb56c0 -1 WARNING: all dangerous and experimental features are enabled.
2025-10-09T05:51:10.276-0500 7f460dd2c6c0 -1 WARNING: all dangerous and experimental features are enabled.
no valid command found; 10 closest matches:
pg stat
pg getmap
pg dump [<dumpcontents:all|summary|sum|delta|pools|osds|pgs|pgs_brief>...]
pg dump_json [<dumpcontents:all|summary|sum|pools|osds|pgs>...]
pg dump_pools_json
pg ls-by-pool <poolstr> [<states>...]
pg ls-by-primary <id|osd.id> [<pool:int>] [<states>...]
pg ls-by-osd <id|osd.id> [<pool:int>] [<states>...]
pg ls [<pool:int>] [<states>...]
pg dump_stuck [<stuckops:inactive|unclean|stale|undersized|degraded>...] [<threshold:int>]
Error EINVAL: invalid command


In [ ]:
%%bash

bin/ceph tell osd.* config set osd_scrub_sleep 0
bin/ceph tell osd.* config set osd_deep_scrub_keys 1

local basefn="/tmp/prmpt_osd_"

# slow-scrub in a loop
for x in {1..100}; do
  echo "Slow-scrub iteration $x"
  # note the last scrub time
  b4=$(bin/ceph pg 1.0 query | jq '.info.history.last_deep_scrub_stamp')
  echo "Last scrub time: $b4"
  bin/ceph tell 1.0 schedule-deep-scrub
  to1=100
  while [ "$b4" == "$(bin/ceph pg 1.0 query | jq '.info.history.last_deep_scrub_stamp')" ]; do
    sleep 0.1
    bin/ceph pg 1.0 query | jq '.scrubber'
    to1=$((to1-1))
    [ $to1 -eq 0 ] && abort
  done
  echo out
  bin/ceph pg 1.0 query | jq '.info.history.last_deep_scrub_stamp'
  #sleep 1
  grep -i preempted out/osd.*.log && break
  #bin/ceph pg 1.0 query | jq '.scrubber'
done

for i in $(seq 0 $((osd_count-1))); do
  echo "$i: into: $basefn$i"
  bin/ceph tell osd.$i counter dump >> "$basefn$i" 2>/dev/null
  bin/ceph tell osd.$i counter dump | grep -E '(scrub_pri)|(scrub_rep)|(scrubs_)|(scrub_)'
done



In [ ]:

def f1(pname, rnds):
  objs = subprocess.check_output(['bin/rados', '-p', pname, 'ls'])
  print(f"objs: {objs}")

  obj_list = objs.decode().split('\n')
  for obj in obj_list:
     print(f"obj: {obj}")
  #ime.sleep(4)
  for x in range(rnds):
    dt1 = subprocess.check_output("date", text=True).strip()
    for obj in (obj for obj in obj_list if re.search(r'.*objec.*', obj)):
      #print(f"obj: {obj}")
      subprocess.run(['echo ' + dt1 + ' | bin/rados -p ' + pname + ' put ' + obj + ' -'], shell=True, check=True)

   #print(f"round {x}")
   #time.sleep(1)


def run_in_thread(f1, pname, rnds):
    """
    Runs the given function `f1` in an async separate thread and passes its parameters.
    :return: An asyncio Future that resolves with the result of `f1`.
    """
    async def run():
        loop = asyncio.get_event_loop()
        with ThreadPoolExecutor() as executor:
            result = await loop.run_in_executor(executor, f1, pname, rnds)
        return result

    return run

async def main1():
   print("main1")
   return run_in_thread(f1, "pl1", 3000)
   

#f1("pl1", 300)

await main1()
#main1()


In [ ]:
%%bash

#cd $CEPH_JTEST_ROOT

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00_b4params.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01_b4params.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02_b4params.json


# set the scheduling parameters

bin/ceph osd pool set pl1 scrub_min_interval 1
bin/ceph osd pool set pl1 scrub_max_interval 2
bin/ceph osd pool set pl1 deep_scrub_interval 1000

#bin/ceph osd pool set pl2 scrub_min_interval 2
#bin/ceph osd pool set pl2 scrub_max_interval 4
#bin/ceph osd pool set pl2 deep_scrub_interval 10

sleep 3

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02.json
#bin/ceph tell osd.3 dump_scrubs --format=json-pretty > /tmp/ds_02.json





In [ ]:
%%bash

#%cd $CEPH_JTEST_ROOT

# common scrub configs
bin/ceph tell osd.* config set osd_blocked_scrub_grace_period 20
bin/ceph tell osd.* config set osd_stats_update_period_scrubbing 2
bin/ceph tell osd.* config set osd_stats_update_period_not_scrubbing 3
#bin/ceph tell osd.* config set osd_scrub_backoff_ratio 0.9999
#bin/ceph tell osd.* config set osd_scrub_interval_randomize_ratio 0.1
#bin/ceph tell osd.* config set osd_deep_scrub_randomize_ratio 0

#bin/ceph tell mgr.$(bin/ceph mgr services | jq -r .mgr) config set mgr_stats_period 2


In [ ]:
%%bash

# list the scrub queue
scrtch=to_"`date +'%H%M%S'`"
echo $scrtch
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json


In [ ]:
%%bash
# set scrub parameters to guarantee slow scrub
bin/ceph tell osd.* config set osd_scrub_sleep "3.0"
bin/ceph tell osd.* config set osd_max_scrubs 1
bin/ceph tell osd.* config set osd_scrub_chunk_max 5
bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 5


In [ ]:
%%bash

# set higher urgency to one of the PGs
bin/ceph tell $pl1_num.7 scrub
bin/ceph tell $pl1_num.6 schedule-deep-scrub
sleep 1
bin/ceph pg dump pgs


In [ ]:
%%bash

scrtch=to_"`date +'%H%M%S'`"
echo $scrtch

# list the scrub queue
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty


In [ ]:
%%bash

# the idea now is to build on the previous attempt, and:
# 1 - create a dictionary of refs to per-pg dictionary of the data (or - if too complex - the data as a list)
# 2 - return multiple objects:
# 1) a full dict as above
# 2) PGs to primary
# 3) PGs to acting set
# 4) pool to PGs
# 5) PG to pool

function build_pg_dicts {
  local dir=$1
  local -n pg_primary_dict=$2
  local -n pg_acting_dict=$3
  local -n pg_pool_dict=$4
  local infile=$5

  local extr_dbg=2 # note: 3 and above leave some temp files around

  #turn off '-x' (but remember previous state)
  local saved_echo_flag=${-//[^x]/}
  set +x

  # if the infile name is '-', fetch the dump directly from the ceph cluster
  if [[ $infile == "-" ]]; then
    local -r ceph_cmd="bin/ceph pg dump pgs_brief -f=json-pretty"
    local -r ceph_cmd_out=$(eval $ceph_cmd)
    local -r ceph_cmd_rc=$?
    if [[ $ceph_cmd_rc -ne 0 ]]; then
      echo "Error: the command '$ceph_cmd' failed with return code $ceph_cmd_rc"
      #return $ceph_cmd_rc
    fi
    (( extr_dbg >= 3 )) && echo "$ceph_cmd_out" > /tmp/e2
    l0=`echo "$ceph_cmd_out" | jq '[.pg_stats | group_by(.pg_stats)[0] | map({pgid: .pgid, pool: (.pgid | split(".")[0]), acting: .acting, acting_primary: .acting_primary})] | .[]' `
  else
    l0=`jq '[.pg_stats | group_by(.pg_stats)[0] | map({pgid: .pgid, pool: (.pgid | split(".")[0]), acting: .acting, acting_primary: .acting_primary})] | .[]' $infile `
  fi
  (( extr_dbg >= 2 )) && echo "L0: $l0"

  mapfile -t l1 < <(echo "$l0" | jq -c '.[]')
  (( extr_dbg >= 2 )) && echo "L1: ${#l1[@]}"

  for item in "${l1[@]}"; do
    pgid=$(echo "$item" | jq -r '.pgid')
    acting=$(echo "$item" | jq -r '.acting | @sh')
    pg_acting_dict["$pgid"]=$acting
    acting_primary=$(echo "$item" | jq -r '.acting_primary')
    pg_primary_dict["$pgid"]=$acting_primary
    pool=$(echo "$item" | jq -r '.pool')
    pg_pool_dict["$pgid"]=$pool
    #pool_dict["$pgid"]="acting=($acting) acting_primary=$acting_primary pool=$pool"
  done

  if [[ -n "$saved_echo_flag" ]]; then set -x; fi
}

# declare -A pg_pr
# declare -A pg_ac
# declare -A pg_po
# build_pg_dicts . pg_pr pg_ac pg_po "-"
# 
# echo "PGs to primary:"
# for pg in "${!pg_pr[@]}"; do
#   echo "Got: $pg: ${pg_pr[$pg]} ( ${pg_ac[$pg]} ) ${pg_po[$pg]}"
# done



# a function that counts the number of common active-set elements between two PGs
# 1 - the first PG
# 2 - the second PG
# 3 - the dictionary of active sets
function count_common_active {
  local pg1=$1
  local pg2=$2
  local -n pg_acting_dict=$3
  local -n res=$4

  local -a a1=(${pg_acting_dict[$pg1]})
  local -a a2=(${pg_acting_dict[$pg2]})

  local -i cnt=0
  for i in "${a1[@]}"; do
    for j in "${a2[@]}"; do
      if [[ $i -eq $j ]]; then
        cnt=$((cnt+1))
      fi
    done
  done

  res=$cnt
}

# a function that returns an array of the common active-set elements between two PGs
# 1 - the first PG
# 2 - the second PG
# 3 - the dictionary of active sets
function get_common_active {
  local pg1=$1
  local pg2=$2
  local -n actng=$3
  local -n res=$4

  local -a a1=(${actng[$pg1]})
  local -a a2=(${actng[$pg2]})

  local -a common=()
  for i in "${a1[@]}"; do
    for j in "${a2[@]}"; do
      if [[ $i -eq $j ]]; then
        common+=($i)
      fi
    done
  done

  res=(${common[@]})
}


# given a PG, find another one with a disjoint active set
# 1 - the PG
# 2 - the dictionary of active sets
# 3 - [out] - the PG with a disjoint active set
function find_disjoint_pg {
  local pg=$1
  local -n ac_dict=$2
  local -n res=$3

  for cand in "${!ac_dict[@]}"; do
    if [[ $cand != $pg ]]; then
      local -i common=0
      count_common_active $pg $cand ac_dict common
      if [[ $common -eq 0 ]]; then
        res=$cand
        return
      fi
    fi
  done
}

# echo "Testing the find_disjoint_pg function"
# rs4=""
# find_disjoint_pg "2.5" pg_ac rs4
# echo "The result: $rs4"

# given a PG, find another one with a disjoint active set
# - but allow a possible common Primary
# 1 - the PG
# 2 - the dictionary of active sets
# 3 - [out] - the PG with a disjoint active set
function find_disjoint_but_primary {
  local pg=$1
  local -n ac_dict=$2
  local -n p_dict=$3
  local -n res=$4

  for cand in "${!ac_dict[@]}"; do
    if [[ "$cand" != "$pg" ]]; then
      local -i common=0
      count_common_active "$pg" "$cand" ac_dict common
      if [[ $common -eq 0 || ( $common -eq 1 && "${p_dict[$pg]}" == "${p_dict[$cand]}" )]]; then
        res=$cand
        return
      fi
    fi
  done
}

# echo "Testing the find_disjoint_but_primary function"
# rs5=""
# find_disjoint_but_primary "2.5" pg_ac pg_pr rs5
# echo "The result: $rs5"


function wait_initial_scrubs() {
    local pg_to_prim_dict=$1
    local extr_dbg=2 # note: 3 and above leave some temp files around
    (( extr_dbg >= 1 )) && echo "waiting initial" && bin/ceph pg dump pgs --format=json-pretty | \
      jq '.pg_stats | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'

    # set a long schedule for the periodic scrubs. Wait for the
    # initial 'no previous scrub is known' scrubs to finish for all PGs.
    bin/ceph tell osd.* config set osd_scrub_min_interval 7200
    bin/ceph tell osd.* config set osd_deep_scrub_interval 14400
    bin/ceph tell osd.* config set osd_max_scrubs 32
    bin/ceph tell osd.* config set osd_scrub_sleep "3.0"

    for pg in "${!pg_to_prim_dict[@]}"; do
      echo "l. 188: <$pg>"
      bin/ceph tell $pg scrub
    done

    (( extr_dbg >= 1 )) && bin/ceph pg dump pgs --format=json-pretty | \
      jq '.pg_stats | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'

    tout=40
    while [ $tout -gt 0 ] ; do
      echo " WAIT $tout"
      sleep 0.5
      #bin/ceph pg dump pgs --format=json-pretty | \
      #  jq '.pg_stats | map(select(.last_scrub_duration == 0)) | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'

      # should this be 'jq -s'? check RRR
      not_done=$(bin/ceph pg dump pgs --format=json-pretty | \
        jq '.pg_stats | map(select(.last_scrub_duration == 0)) | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})' | wc -l )
      # note that we should ignore a header line
      if [ "$not_done" -le 1 ]; then
        break
      fi
      not_done=$((not_done - 1))

      echo "Still waiting for $not_done PGs to finish initial scrubs"
      tout=$(($tout - 1))
    done

    (( extr_dbg >= 1 )) && bin/ceph pg dump pgs --format=json-pretty | \
      jq '.pg_stats | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'
    (( tout == 0 )) && return 1
}


function get_asok_dir() {
    local CEPH_ASOK_DIR=$(bin/ceph-conf --lookup asok_dir)
    if [ -n "$CEPH_ASOK_DIR" ]; then
        echo "$CEPH_ASOK_DIR"
    else
        echo ${TMPDIR:-/tmp}/ceph-asok.$$
    fi
}

function get_asok_path() {
    local name=$1
    echo /home/rfriedma/src/k3/ceph/build/asok/ceph-$name.asok
#     if [ -n "$name" ]; then
#         echo $(get_asok_dir)/ceph-$name.asok
#     else
#         echo $(get_asok_dir)/\$cluster-\$name.asok
#     fi
}

function set_query_debug() {
    local pgid=$1
    local prim_osd=`bin/ceph pg dump pgs_brief | \
      awk -v pg="^$pgid" -n -e '$0 ~ pg { print(gensub(/[^0-9]*([0-9]+).*/,"\\\\1","g",$5)); }' `

    echo "Setting scrub debug data. Primary for $pgid is $prim_osd"
    get_asok_path osd.$prim_osd
    echo "Setting scrub debug data. Primary for $pgid is $prim_osd"
    CEPH_ARGS='' bin/ceph --format=json daemon $(get_asok_path osd.$prim_osd) \
          scrubdebug $pgid set sessions
}




function TEST_abort_periodic_for_operator() {
    local dir=$1
    local -A cluster_conf=(
        ['osds_num']="6" 
        ['pgs_in_pool']="16"
        ['pool_name']="test"
    )
    set -x

    #standard_scrub_wpq_cluster "$dir" cluster_conf 3 || return 1
    #local poolid=${cluster_conf['pool_id']}
    #local poolname=${cluster_conf['pool_name']}


    #modified for the Jupyter environment

   # the cluster was already created

    local poolid=$pl1_num
    local poolname="pl1"
    echo "Pool: $poolname : $poolid"

    set +x
    # fill the pool with some data
    local TESTDATA="/tmp/testdata.$$"
    dd if=/dev/urandom of="$TESTDATA" bs=1032 count=1
    for i in $( seq 1 25 )
    do
        bin/rados -p "$poolname" put "obj${i}" "$TESTDATA" 2>&1 1>/dev/null
    done
    rm -f "$TESTDATA"


    # create the dictionary of the PGs in the pool
    declare -A pg_pr
    declare -A pg_ac
    declare -A pg_po
    build_pg_dicts "$dir" pg_pr pg_ac pg_po "-"

    echo "PGs data:"
    for pg in "${!pg_pr[@]}"; do
      echo "Got: $pg: ${pg_pr[$pg]} ( ${pg_ac[$pg]} ) ${pg_po[$pg]}"
    done

    for pg in "${!pg_pr[@]}"; do
      echo "bin/ceph tell $pg scrub"
      bin/ceph tell $pg scrub || return 1
    done
    set -x

    wait_initial_scrubs pg_pr

    bin/ceph tell osd.2 dump_scrub_reservations --format=json-pretty
    # limit all OSDs to one scrub at a time
    bin/ceph tell osd.* config set osd_max_scrubs 1

    # configure for slow scrubs
    bin/ceph tell osd.* config set osd_scrub_sleep 3
    bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 2
    bin/ceph tell osd.* config set osd_scrub_chunk_max 2

    # the first PG to work with:
    local pg1="1.0"
    # and another one, that shares its primary, and at least one more active set member
    local pg2=""
     for pg in "${!pg_pr[@]}"; do
      if [[ "${pg_pr[$pg]}" == "${pg_pr[$pg1]}" ]]; then
        local -i common=0
        count_common_active $pg $pg1 pg_ac common
        if [[ $common -gt 1 ]]; then
          pg2=$pg
          break
        fi
      fi
    done
    if [[ -z "$pg2" ]]; then
      # \todo handle the case when no such PG is found
      echo "No PG found with the same primary as $pg1"
      return 1
    fi

    echo "The primary (${pg_pr[$pg1]}) is allowed two concurrent scrubs"
    bin/ceph tell osd."${pg_pr[$pg1]}" config set osd_max_scrubs 2
    echo "=xxx==================== $pg1 ================== $pg2 =============================="
    # collect the timestamps before issuing the scrub command
    set_query_debug "$pg1"
    echo "<<1>>>: query:"
    echo
    bin/ceph pg "$pg1" query
    before_stamp=$(bin/ceph pg "$pg1" query | jq '.info.stats.last_deep_scrub_stamp')
    bin/ceph tell $pg1 schedule-deep-scrub

    sleep 1
    echo ' after 1 second'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber.active'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber.is_reserving_replicas'

    for i in $( seq 1 10 )
    do
      sleep 0.5
      stt=$(bin/ceph pg "$pg1" query | jq '.scrubber')
      is_active=$(echo $stt | jq '.active')
      is_reserving_replicas=$(echo $stt | jq '.is_reserving_replicas')
      if [[ "$is_active" = "true" && "$is_reserving_replicas" = "false" ]]; then
          break
      fi
      echo "Still waiting: $stt"
    done
    if [[ "$is_active" != "true" || "$is_reserving_replicas" != "false" ]]; then
      echo "The scrub is not active or is reserving replicas"
      return 1
    fi

    #sleep 1
    # make sure the scrub is in progress (past the registration stage)
    #while [ $(bin/ceph pg "$pg1" query | f json-pretty | jq '.scrubber.'  ) -gt 0 ] ; do

    #echo ' after 4 second'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber'
    #bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber.active'
    #bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber.is_reserving_replicas'


    echo "===================================================================================="

    bin/ceph tell osd.2 dump_scrub_reservations --format=json-pretty

    # now - the 2'nd scrub - which should be blocked on reserving
    set_query_debug "$pg2"
    bin/ceph tell "$pg2" schedule-deep-scrub
    sleep 0.5
    bin/ceph tell 1.6 schedule-deep-scrub
    bin/ceph tell 1.1 schedule-deep-scrub

    bin/ceph tell osd.2 dump_scrub_reservations --format=json-pretty
    bin/ceph tell osd.1 dump_scrub_reservations --format=json-pretty

    echo
    echo "<<2>>>: query:"
    echo
    #bin/ceph pg "$pg2" query
    echo "===================================================================================="
    bin/ceph pg "$pg2" query -f json-pretty | jq '.scrubber'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber'
    sleep 2
    echo "===================================================================================="
    bin/ceph pg "$pg2" query -f json-pretty | jq '.scrubber'
    bin/ceph pg "$pg1" query -f json-pretty | jq '.scrubber'

    # make sure pg2 scrub is stuck in the reserving state
    local stt2=$(bin/ceph pg "$pg2" query | jq '.scrubber')
    local pg2_is_reserving
    pg2_is_reserving=$(echo $stt2 | jq '.is_reserving_replicas')
    if [[ "$pg2_is_reserving" != "true" ]]; then
      echo "The scheduled scrub for $pg2 should have been stuck"
      return 1
    fi

    # now - issue an operator-initiated scrub on pg2.
    # The periodic scrub should be aborted, and the operator-initiated scrub should start.
    bin/ceph tell "$pg2" scrub
    for i in $( seq 1 10 )
    do
      sleep 0.5
      stt2=$(bin/ceph pg "$pg2" query | jq '.scrubber')
      pg2_is_active=$(echo $stt2 | jq '.active')
      pg2_is_reserving=$(echo $stt2 | jq '.is_reserving_replicas')
      if [[ "$pg2_is_active" = "true" && "$pg2_is_reserving" != "true" ]]; then
            break
      fi
      echo "Still waiting: $stt2"
    done

    if [[ "$pg2_is_active" != "true" || "$pg2_is_reserving" = "true" ]]; then
      echo "The high-priority scrub for $pg2 is not active or is reserving replicas"
      return 1
    fi

#     do
#       sleep 0.5
#       echo "PG1 
#       after_stamp=$(bin/ceph pg "$pg1" query | jq '.info.stats.last_deep_scrub_stamp')
#       echo "Before: $before_stamp, After: $after_stamp"
#       [[ "$before_stamp" != "$after_stamp" ]] && break
#       # \todo add a timeout
#     done


#     # find two disjoint PGs
#     local pg1="1.0"
#     local pg2=""
#     find_disjoint_pg "$pg1" pg_ac pg2
#     echo "Totally disjoint: $pg1 and $pg2"
# 
#     local pg3="1.e"
#     local pg4=""
#     find_disjoint_but_primary "$pg3" pg_ac pg_pr pg4
#     echo "Same primary allowed: $pg3 and $pg4"
}

TEST_abort_periodic_for_operator "."


echo "done"

In [ ]:
%%bash

# finding out that all PGs were scrubbed at least once
bin/ceph pg dump pgs --format=json-pretty | jq '.pg_stats | map(select(.last_scrub_duration != "0")) | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})'
bin/ceph pg dump pgs --format=json-pretty | jq '.pg_stats | map(select(.last_scrub_duration == "0")) | map({pgid: .pgid, last_scrub_duration: .last_scrub_duration})' | wc -l


In [ ]:
raise SystemExit("Stop here")

# Termination


In [ ]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
cd $CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    cd $jp
    %env CEPH_JTEST_ROOT=$jp
fi

pwd

../src/stop.sh
sleep 4
../src/stop.sh
